## Google Colab Setup

**GPU Required:** Before running, enable GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** (or better)
3. Click **Save**

In [1]:
# Install dependencies (skip if already done)
import os

# Set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

if not os.path.isfile("FOUNDRY_READY"):
    print("Installing rc-foundry...")

    # Uninstall torchvision first to avoid operator conflicts
    os.system("pip uninstall -y torchvision")

    # Install rc-foundry
    os.system("pip install -q 'rc-foundry[all]'")

    # Mark as ready
    os.system("touch FOUNDRY_READY")

    print("Done!")
else:
    print("rc-foundry already installed.")

Installing rc-foundry...
Done!


In [3]:
# Download model weights (skips already-downloaded models automatically)
# ~3GB RFD3 + MPNN; Boltz2 checkpoint is fetched in Section 3 (or set BOLTZ2_CKPT)
os.system("foundry install rfd3 ligandmpnn")

# Install boltz2
os.system("pip install boltz[cuda] -U")

!pip install -U --force-reinstall --no-cache-dir "numpy<2"

0

# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Generation** | RFD3 | Generate novel proteins via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation via Refolding** | Boltz2 | Predict the structure from designed sequence to validate designability |

Design and MPNN stages use [AtomWorks](https://github.com/RosettaCommons/atomworks) / Biotite `AtomArray` objects. Refolding uses the **Boltz2 command line** (`boltz predict`) with YAML inputs. With `--use_msa_server`, MSAs are generated via the ColabFold/MMseqs2 API for each run; for large batches you can instead precompute `.a3m` paths in YAML and omit that flag (see [Boltz prediction docs](https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md)).

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → Boltz2 (validation) → RMSD comparison
```

---

In [9]:
# find contacts between target and binder for motif scaffolding
input_pdb = "/content/fold_basic_invasin_model_0.cif"

binder_chain = "A"
target_chain = "B,C"
target_sequences = ""

binder_residues = []
target_residues = []



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 184.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
access 1

## Section 1: All-Atom Generation with RFD3

RFdiffusion3 (RFD3) generates *de novo* all-atom proteins that meet specific conditioning requirements.

**Parameters Used** *(many more are available for more complex protein design tasks)*:
- `length`: Target protein length in residues
- `diffusion_batch_size`: Number of structures to generate per batch
- `n_batches`: Number of batches to run

**Outputs:** Dictionary of `RFD3Output` objects.

In [1]:
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

# Set seed for reproducibility
seed_everything(0)

# Configure RFD3 inference
config = RFD3InferenceConfig(
    specification={
        'input': "/content/fold_basic_invasin_model_0.cif",
        'contig': "30-50,C113-120,30-80,/0,A1-377,/0,B1-480",
        'select_hotspots': "A70,A256,B160",
        'infer_ori_strategy': 'hotspots',
    },
    diffusion_batch_size=2,  # Generate 2 structures per batch
)

# Initialize engine and run generation
model = RFD3InferenceEngine(**config)
outputs = model.run(
    inputs=None,      # None for unconditional generation
    out_dir=None,     # None to return in memory (no file output)
    n_batches=2,      # Generate 1 batch
)

  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
INFO:foundry:cuEquivariance is available and will be used.
DEBUG:transforms:Debug mode is on
INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: You

In [2]:
# Inspect RFD3 outputs and extract the generated structures
#for idx, data in outputs.items():
#    print(f"Batch {idx}: {len(data)} structure(s)")
#    print(f"  Output type: {type(data[0]).__name__}")
#    print(f"  AtomArray: {data[0].atom_array}")

# Extract the first generated structure for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

# Visualize the generated structure
view(atom_array)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [18]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": None,        # Return results in memory
    "write_structures": False,
    "write_fasta": False,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 8,         # Generate 10 sequences per structure
        "remove_waters": True,
        "designed_chains": ["A"],
        #"fixed_chains": ["B","C"],
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = []
for first_key in outputs.keys():
  atom_array = outputs[first_key][0].atom_array
  mpnn_outputs.append([model.run(input_dicts=input_configs, atom_arrays=[atom_array]),int(first_key[1:])])

*(YAML inputs are written in the next code cell — one `*.yaml` per MPNN design, then `boltz predict` is run.)*

---

## Section 3: Structure Prediction with Boltz2 (CLI)

This section writes a valid **[Boltz YAML](https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md)** (version + `sequences` list) for each MPNN design and runs:

`boltz predict <job>.yaml --out_dir <job_dir> --model boltz2 --use_msa_server ...`

Under `<job_dir>/predictions/<job>/` you get `<job>_model_0.cif` (or `.pdb`), `confidence_<job>_model_0.json`, etc.

**Protein-only here:** chains are inferred from standard amino-acid residues in `item.atom_array`. Add `ligand` / `ccd` blocks to the YAML if your designs include small molecules.

**Tip:** Use a **separate `--out_dir` per design** (`…/boltz_runs/1_1`) so cached outputs do not skip reruns; `--override` forces a fresh prediction.

In [6]:
"""Boltz2 refold via CLI: per-design YAML → `boltz predict` → CIF + confidence JSON."""

from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
from biotite.sequence import ProteinSequence
from biotite.structure import get_residue_starts
from biotite.structure import io as strucio

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml

# ── knobs ──────────────────────────────────────────────────────────
BOLTZ_RUN_ROOT = Path("/content/boltz_refold")  # change for local runs
USE_MSA_SERVER = True  # False → each protein must have `msa: path/to.a3m` or `empty`
BOLTZ_ACCELERATOR = "gpu"  # "cpu" if no GPU
BOLTZ_NO_KERNELS = False  # True fixes some older NVIDIA + cuequivariance errors
# ───────────────────────────────────────────────────────────────────

BOLTZ_RUN_ROOT.mkdir(parents=True, exist_ok=True)

_boltz = shutil.which("boltz")
if _boltz is None:
    raise RuntimeError('Install Boltz first (e.g. pip install "boltz[cuda]" -U).')


def protein_sequences_by_chain(atom_array) -> dict[str, str]:
    """One-letter sequences for each chain (standard amino acids only)."""
    sequences: dict[str, str] = {}
    for cid in np.unique(atom_array.chain_id):
        mask = atom_array.chain_id == cid
        sub = atom_array[mask]
        starts = get_residue_starts(sub)
        try:
            seq = "".join(
                ProteinSequence.convert_letter_3to1(sub.res_name[starts])
            )
        except (KeyError, TypeError, ValueError):
            continue
        if seq:
            sequences[str(cid)] = seq
    if not sequences:
        raise ValueError("No standard protein chains found on atom_array.")
    return sequences


def build_boltz_yaml(chain_sequences: dict[str, str], *, use_msa_server: bool) -> dict:
    """Valid Boltz YAML: `version` + `sequences` list.

    Use mapping keys like protein:, not list items like '- protein:'.
    If `use_msa_server` is True, omit `msa` and pass `--use_msa_server` to the CLI.
    """
    entries = []
    for cid in sorted(chain_sequences.keys(), key=lambda x: (len(x), x)):
        block: dict = {"id": cid, "sequence": chain_sequences[cid]}
        if not use_msa_server:
            block["msa"] = "empty"
        entries.append({"protein": block})
    return {"version": 1, "sequences": entries}


boltz_rows = []
for mpnn_output_x in mpnn_outputs:
    for i, item in enumerate(mpnn_output_x[0]):
        ex_id = f"{mpnn_output_x[1] + 1}_{i + 1}"
        aa = item.atom_array

        chain_seqs = protein_sequences_by_chain(aa)
        res_starts = get_residue_starts(aa)
        seq_concat = "".join(
            ProteinSequence.convert_letter_3to1(aa.res_name[res_starts])
        )
        print(f"Job {ex_id}: chains {list(chain_seqs.keys())}  (concat len {len(seq_concat)})")

        run_dir = BOLTZ_RUN_ROOT / ex_id
        run_dir.mkdir(parents=True, exist_ok=True)
        yaml_path = run_dir / f"{ex_id}.yaml"
        yaml_body = build_boltz_yaml(chain_seqs, use_msa_server=USE_MSA_SERVER)
        yaml_path.write_text(yaml.safe_dump(yaml_body, sort_keys=False, default_flow_style=False))

        cmd = [
            _boltz,
            "predict",
            str(yaml_path),
            "--out_dir",
            str(run_dir),
            "--model",
            "boltz2",
            "--accelerator",
            BOLTZ_ACCELERATOR,
            "--output_format",
            "mmcif",
            "--override",
        ]
        if USE_MSA_SERVER:
            cmd.append("--use_msa_server")
        if BOLTZ_NO_KERNELS:
            cmd.append("--no_kernels")

        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)

        pred_dir = run_dir / "predictions" / ex_id
        conf_path = pred_dir / f"confidence_{ex_id}_model_0.json"
        if not conf_path.is_file():
            raise FileNotFoundError(
                f"Missing {conf_path}. Check Boltz logs under {run_dir}."
            )

        struct_paths = list(pred_dir.glob(f"{ex_id}_model_0.cif"))
        if not struct_paths:
            struct_paths = list(pred_dir.glob(f"{ex_id}_model_0.pdb"))
        if not struct_paths:
            raise FileNotFoundError(
                f"No {ex_id}_model_0.cif/.pdb under {pred_dir}"
            )

        with conf_path.open() as f:
            scores = json.load(f)

        atom_pred = strucio.load_structure(str(struct_paths[0]))

        boltz_rows.append(
            {
                "example_id": ex_id,
                "name": ex_id,
                "rfd3_num": int(ex_id.split("_")[0]),
                "mpnn_num": int(ex_id.split("_")[1]),
                "sequence": seq_concat,
                "atom_array": atom_pred,
                "overall_plddt": float(scores["complex_plddt"]),
                "ptm": float(scores["ptm"]),
                "iptm": float(scores["iptm"]),
                "confidence_score": float(scores["confidence_score"]),
            }
        )

ImportError: foundry.model_utils is required. Install from a checkout that includes src/foundry/model_utils.py (e.g. pip install -e . in the foundry repo), or add that directory to PYTHONPATH.

In [ ]:
import pandas as pd

rf3_df = pd.DataFrame(boltz_rows)

print("Boltz2 refold DataFrame created successfully!")
print(rf3_df.head())
print("\nDataFrame columns:", rf3_df.columns.tolist())

In [ ]:
# Extract the top-ranked prediction
rf3_output = rf3_df.loc[rf3_df['overall_plddt'].idxmax()]

# Visualize the predicted structure
view(rf3_output["atom_array"])

---

## Section 4: Validation and Export

The final step compares the Boltz2-predicted structure against the original RFD3-generated backbone. A low backbone RMSD indicates the designed sequence is likely to fold into the intended structure (high designability).

In [ ]:
from biotite.structure import rmsd, superimpose
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
import numpy as np
from atomworks.io.utils.io_utils import to_cif_file

for index, row in rf3_df.iterrows():
  # Get structures for comparison
  aa_generated = outputs["_"+str(int(row['rfd3_num'])-1)][0].atom_array              # Original RFD3 backbone (from Section 1)
  aa_refolded = row["atom_array"]    # Boltz2-predicted structure

  # Filter to backbone atoms (N, CA, C, O)
  bb_generated = aa_generated[np.isin(aa_generated.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]
  bb_refolded = aa_refolded[np.isin(aa_refolded.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]

  # Superimpose structures and calculate RMSD
  bb_refolded_fitted, _ = superimpose(bb_generated, bb_refolded)
  rmsd_value = rmsd(bb_generated, bb_refolded_fitted)
  rf3_df.loc[index,"rmsd"] = rmsd_value

  print(f"\nBackbone RMSD: {rmsd_value:.2f} A")
  print(f"Interpretation: {'Excellent' if rmsd_value < 1.0 else 'Good' if rmsd_value < 2.0 else 'Moderate'} designability")
  if rmsd_value < 2.0:
    to_cif_file(aa_refolded,row['example_id']+"_refolded.cif")

# export rf3_df to csv
rf3_df.to_csv("all_designs.csv")